# Lecture 2: Answers to the practice questions
Answers to Practice 1–2 from the slides. Requires `spacy`, `en_core_web_sm` and `unidecode` (see the setup slide in Lecture 1).

## Practice 1: Clean by hand
Run the headlines through the `clean_strings()` function from the slides.

In [1]:
import string
import unidecode

def clean_strings(x):
    x = x.lower()                                               # To lower
    x = x.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    x = unidecode.unidecode(x)                                  # ASCII folding
    x = " ".join(x.split())                                     # Normalize whitespace
    return x

headlines = [
    "  Ørsted shares fall 12% after U.S. wind project is halted!  ",
    "$NVO up 3.2% pre-market — analysts: 'not a disaster'",
    "Mærsk: NOT a good quarter.",
]

for h in headlines:
    print(repr(clean_strings(h)))

'orsted shares fall 12 after us wind project is halted'
'nvo up 32 premarket -- analysts not a disaster'
'maersk not a good quarter'


**What was lost?**
1. The `%` and the `!` are gone, and "U.S." became "us", which is now the pronoun.
2. "3.2%" became "32": **a different number**. "$NVO" lost its ticker marker. The em dash "—" is not in `string.punctuation`, so it survives the punctuation step, and `unidecode` then turns it into `--`.
3. The emphasis in "NOT" is lost, but the word itself is kept.

**The order of the steps matters.** If we fold to ASCII *before* removing punctuation, the dash is removed as well:

In [2]:
def clean_strings_v2(x):
    x = unidecode.unidecode(x)                                  # ASCII folding first
    x = x.lower()
    x = x.translate(str.maketrans('', '', string.punctuation))
    x = " ".join(x.split())
    return x

print(repr(clean_strings_v2(headlines[1])))

'nvo up 32 premarket analysts not a disaster'


**Which step would you skip for a sentiment model?** Lowercasing ("NOT", "US") and removing numbers and punctuation ("3.2%", "!"). They throw away exactly the signals that matter.

## Practice 2: Predict spaCy
*"Novo Nordisk did not beat expectations, and shares fell."*

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("Novo Nordisk did not beat expectations, and shares fell.")

# 1. Lemmas
for token in doc:
    print(f"{token.text:>14} -> {token.lemma_}")

          Novo -> Novo
       Nordisk -> Nordisk
           did -> do
           not -> not
          beat -> beat
  expectations -> expectation
             , -> ,
           and -> and
        shares -> share
          fell -> fall
             . -> .


In [4]:
# 2. Which tokens are stopwords?
print("Stopwords: ", [t.text for t in doc if t.is_stop])

# 3. What is left?
print("Left:      ", " ".join(t.text for t in doc if not t.is_stop))

Stopwords:  ['did', 'not', 'and']
Left:       Novo Nordisk beat expectations , shares fell .


1. did → do, shares → share, fell → fall.
2. spaCy drops "did", "not" and "and".
3. What is left reads *"Novo Nordisk beat expectations, shares fell"*. **The negation is gone, and the headline now sounds like good news.** Removing stopwords can flip the sentiment.